# TFG: Discos Duros en la Superficie de un Cubo: ¿Una Vía Hacia la Frustración Geométrica en Dos Dimensiones?

Alumno: Hugo Amilibia Ramírez

Tutor: Giorgio Cinacchi


Este notebook será una recopilación general de las distintas funciones que voy a ir haciendo y usando. Incluyo una breve explicacion sobre lo que hacen y puntualizaciones que considere necesarias. 

Además, creo que una buena manera de comprobar si funcionan correctamente, sobre todo las de la parte del análisis, es adaptando todo a la topología de un plano normal con PBC. Ya conocemos los resultados de las distintas métricas y parámetros de orden para este caso, por lo que es una manera de blindar el motor y el código del análisis frente a errores o 'falsos positivos'. Será un poco tedioso ya que para la mayoría de funciones tendré que hacer dos versiones. Las voy a diferenciar usando el parámetro topology, donde un 0 será para el plano normal y un 1 será para el cubo desdoblado: topology==0 (plano), topology==1 (cubo). Las dimensiones correspondientes son L x L para el plano y el cubo también tiene lado L, es decir, desdoblado nos da una cruz de seis planos, todos ellos de lado L.

# Nota: 
Se me ha olvidado poner con que librerías va cada función, lo haré más adelante

# PBC y definición de las geometrías
Aquí tengo las PBC necesarias para cada topología. He comprobado en el script 'prueba pbc.py' que el desdoble 2D del cubo realmente se comporte como el cubo real en 3D; en este script la función también incluye una parte para actualizar las velocidades (se explica todo en el .py). Esta es la versión modificada para actualizar solo las posiciones.

In [ ]:

@njit(fastmath=True)
def inside(x, y, L, topology):
    
    if topology == 0:
        if 0 <= x <= L and 0 <= y <= L: 
            return True
        return False
    
    if topology == 1:    
        if L <= x <= 2*L and 0 <= y <= 4*L: return True
        if 0 <= x <= 3*L and L <= y <= 2*L: return True
    return False

@njit(fastmath=True)
def PBC(old_x, old_y, new_x, new_y, L, topology):
    
    if topology == 0:
        return new_x % (L), new_y % (L)
        
    
    if topology == 1:
        if inside(new_x, new_y, L, topology): return new_x, new_y
            
        if old_y < L: 
            if new_y < 0: return new_x, new_y + 4*L               
            if new_x < L: return new_y, 2*L - new_x               
            if new_x > 2*L: return 3*L - new_y, new_x - L         
        elif old_y > 3*L: 
            if new_y > 4*L: return new_x, new_y - 4*L             
            if new_x < L: return L - new_x, 5*L - new_y           
            if new_x > 2*L: return 5*L - new_x, 5*L - new_y       
        elif old_x < L: 
            if new_y < L: return 2*L - new_y, new_x               
            if new_x < 0: return L - new_x, 5*L - new_y           
            if new_y > 2*L: return new_y - L, 3*L - new_x         
        elif old_x > 2*L: 
            if new_y < L: return L + new_y, 3*L - new_x           
            if new_x > 3*L: return 5*L - new_x, 5*L - new_y       
            if new_y > 2*L: return 4*L - new_y, new_x             
        else: 
            if 2*L <= old_y <= 3*L: 
                if new_x < L: return 3*L - new_y, L + new_x       
                if new_x > 2*L: return new_y, 4*L - new_x         
                    
        return new_x, new_y
    

# Distancias
Luego por el uso de las PBC o de la geometría del propio cubo, las distancias no siempré serán las euclídeas, si no que hay que usar las geodésicas. Para la parte del plano simplemente uso la convención de imagén mínima (MIC) y para el cubo he definido la distancia que habría entre cada par de caras. Recordar que esta manera de escribir la MIC es mejor con Numba que la manera que usé en el proyecto de IA con operaciones round y dot. Numba premia las operaciones simples. De momento esto no se usa para nada. Será útil más adelante con todo el tema del análisis, donde si es necesario el cálculo de las distancias de esta manera, ya que para las simulaciones es todo con el Cells List

In [ ]:

@njit(fastmath=True)
def dist_g(x1, y1, x2, y2, L, topology):
    
    if topology == 0:
        # Convención de mínima imagen para el plano
        dx=abs(x1-x2)
        dy=abs(y1-y2)
        
        if dx > 0.5 * L: dx = L - dx
        if dy > 0.5 * L: dy = L - dy
        
        return np.sqrt(dx**2 + dy**2)
    
    if topology == 1:
     min_d2 = (x1 - x2)**2 + (y1 - y2)**2 
    
     # 4 - 1 
     d2 = (x1 - x2)**2 + (y1 - (y2 - 4*L))**2
     if d2 < min_d2: min_d2 = d2
     d2 = (x1 - x2)**2 + (y1 - (y2 + 4*L))**2
     if d2 < min_d2: min_d2 = d2
        
     # 5 - 1 
     d2 = (x1 - (2*L - y2))**2 + (y1 - x2)**2
     if d2 < min_d2: min_d2 = d2
     d2 = (x1 - y2)**2 + (y1 - (2*L - x2))**2
     if d2 < min_d2: min_d2 = d2
        
     # 6 - 1 
     d2 = (x1 - (L + y2))**2 + (y1 - (3*L - x2))**2
     if d2 < min_d2: min_d2 = d2
     d2 = (x1 - (3*L - y2))**2 + (y1 - (x2 - L))**2
     if d2 < min_d2: min_d2 = d2
        
     # 3 - 5 
     d2 = (x1 - (3*L - y2))**2 + (y1 - (L + x2))**2
     if d2 < min_d2: min_d2 = d2
     d2 = (x1 - (y2 - L))**2 + (y1 - (3*L - x2))**2
     if d2 < min_d2: min_d2 = d2
        
     # 3 - 6 
     d2 = (x1 - y2)**2 + (y1 - (4*L - x2))**2
     if d2 < min_d2: min_d2 = d2
     d2 = (x1 - (4*L - y2))**2 + (y1 - x2)**2
     if d2 < min_d2: min_d2 = d2
        
     # 4 - 5 
     d2 = (x1 - (L - x2))**2 + (y1 - (5*L - y2))**2
     if d2 < min_d2: min_d2 = d2
        
     # 4 - 6 
     d2 = (x1 - (5*L - x2))**2 + (y1 - (5*L - y2))**2
     if d2 < min_d2: min_d2 = d2
        
    return np.sqrt(min_d2)

# Interacciones a través de los bordes
Ahora, tanto para la parte viusal de la simulación como para que los discos interactuén correctamente es necesario definir donde están exactamente teniendo en cuenta que pueden 'estar en varios sitios a la vez'. Para esto he hecho la función 'get_ghosts'. Aquí, en vez de calcular  donde una regla de colisión especial comprobando donde está el disco en todo momento la simulación duraría una barbaridad.

Lo que intento hacer aquí (y creo que funciona) es que si un disco se acerca a un borde, se generen las coordenadas de un "fantasma" de ese disco en la cara correspondiente. Simplemente son transformaciones rígidas. 

De esta manera, solo tengo que tratar a los fantasmas como si fueran partículas normales en el plano 2D local, ignorando toda la topología compleja del cubo 3D.

La comprobación también está hecha en el script 'prueba pbc.py', pero en este lo hice inicialmente como un .append, es decir, una lista dinámica. Entonces, lo he cambiado por una matriz fija con el np.full y así optiizar el rendimiento con numba. Pasando de listas dinámicas a arrays fijos. Esto es importante porque numba no soporta bien las listas dinámicas y el uso de np.full permite preasignar memoria para un número fijo de fantasmas, lo que supuestamente mejora la eficiencia. Esta matriz recoje las posiciones de los fantasmas con un contador (idx), sus dimensiones admiten hasta 8 fantasmas simplemente por seguridad y no quedarme corto, pero si las cuentas no me fallan el valor máximo dbería ser 4. El -100 es simplemente para que si un disco no necesita fantasmas, la matriz devuelva -100.0, y entonces simplemente se ignoran esas coordenadas al evaluar las colisiones. 

En principio esto funciona, y está optimizado, pero todavía tengo ciertas dudas sobre ello.

Finalmente para esta parte, quiero acordarme de que el valor de threshold no me queda claro, es decir, en el script inicila lo escribí como el radio R directamente, pero no se si debería ser el diámetro D, por eso lo pongo así de momento. Realmente tomé la decisión de poner R porque el motivo principal por el que hice la función era para la visualización, ya que en ese script utilizo solo un disco de prueba, y al final me parece la medida adecuada para empezar a dibujar el disco en la cara adyacente, pero igual tengo que poner D para las colisiones y R para la visualización.

In [ ]:
@njit(fastmath=True)
def get_ghosts(x, y, L, threshold, topology):
    ghosts = np.full((14, 2), -100.0) 
    idx = 0
    
    if topology == 0:
        if x < threshold: ghosts[idx] = (x + L, y); idx+=1
        elif x > L - threshold: ghosts[idx] = (x - L, y); idx+=1
        if y < threshold: ghosts[idx] = (x, y + L); idx+=1
        elif y > L - threshold: ghosts[idx] = (x, y - L); idx+=1
        if x < threshold and y < threshold: ghosts[idx] = (x + L, y + L); idx+=1
        elif x > L - threshold and y < threshold: ghosts[idx] = (x - L, y + L); idx+=1
        elif x < threshold and y > L - threshold: ghosts[idx] = (x + L, y - L); idx+=1
        elif x > L - threshold and y > L - threshold: ghosts[idx] = (x - L, y - L); idx+=1
        return ghosts
    
    if topology == 1:
        # 1 abajo - 4 arriba
        if y < threshold and L <= x <= 2*L: ghosts[idx] = (x, y + 4*L); idx+=1
        if y > 4*L - threshold and L <= x <= 2*L: ghosts[idx] = (x, y - 4*L); idx+=1
        
        # 1 izquierda - 5 abajo
        if x < L + threshold and x > L and y < L: ghosts[idx] = (y, 2*L - x); idx+=1
        if y < L + threshold and y > L and x < L: ghosts[idx] = (2*L - y, x); idx+=1
        
        # 1 derecha - 6 abajo
        if x > 2*L - threshold and x < 2*L and y < L: ghosts[idx] = (3*L - y, x - L); idx+=1
        if y < L + threshold and y > L and x > 2*L: ghosts[idx] = (L + y, 3*L - x); idx+=1
        
        # 3 izquierda - 5 arriba
        if x < L + threshold and x > L and 2*L < y < 3*L: ghosts[idx] = (3*L - y, L + x); idx+=1
        if y > 2*L - threshold and y < 2*L and x < L: ghosts[idx] = (y - L, 3*L - x); idx+=1
        
        # 3 derecha - 6 arriba
        if x > 2*L - threshold and x < 2*L and 2*L < y < 3*L: ghosts[idx] = (y, 4*L - x); idx+=1
        if y > 2*L - threshold and y < 2*L and x > 2*L: ghosts[idx] = (4*L - y, x); idx+=1
        
        # 4 izquierda - 5 izquierda 
        if x < L + threshold and x > L and y > 3*L: ghosts[idx] = (L - x, 5*L - y); idx+=1
        if x < threshold and L < y < 2*L: ghosts[idx] = (L - x, 5*L - y); idx+=1 
        
        # 4 derecha - 6 derecha
        if x > 2*L - threshold and x < 2*L and y > 3*L: ghosts[idx] = (5*L - x, 5*L - y); idx+=1
        if x > 3*L - threshold and L < y < 2*L: ghosts[idx] = (5*L - x, 5*L - y); idx+=1 
        
    return ghosts

# Colisiones
En principio ya tengo como se comporta la geometría en su totalidad, ahora necesito un motor de colisiones que implemente esa geometría Para ello tengo las siguientes funciones, que están hechas basándome en un algoritmo de cell lists para ver si optimizamos al máximo. Por un lado tengo una función para el chekeo local de cada celda (check_grid), y por otro tenemos el chekeo a través de los bordes.  

Para que esto funcione bien utilizamos las coordenadas de los centroides se chekean bloques de 9 celdas, una central y 8 exteriores.

In [ ]:
@njit(fastmath=True)
def check_grid(px, py, x, y, D, head, next_node, ncx, ncy, ignore_idx):
    
    cx, cy = int(px/D), int(py/D)
    for dcx in (-1, 0, 1):
        for dcy in (-1, 0, 1):
            ccx = cx + dcx
            ccy = cy + dcy
            if 0 <= ccx < ncx and 0 <= ccy < ncy:
                j = head[ccx + ccy * ncx]
                while j != -1:
                    if j != ignore_idx:
                        if (px - x[j])**2 + (py - y[j])**2 < D*D:
                            return True
                    j = next_node[j]
    return False



@njit(fastmath=True)
def check_overlap(px, py, x, y, D, head, next_node, ncx, ncy, L, topology, ignore_idx=-1):

    if check_grid(px, py, x, y, D, head, next_node, ncx, ncy, ignore_idx): 
        return True
    
    ghosts = get_ghosts(px, py, L, D, topology)
    for i in range(8):
        gx, gy = ghosts[i]
        if gx != -100.0: 
            if check_grid(gx, gy, x, y, D, head, next_node, ncx, ncy, ignore_idx): 
                return True
                
    return False



# Inicialización y montecarlo principal (MC traslacional)

Con el motor de colisiones y la geometría completos, podemos empezara a plantear la inicialización de la simulación. Para ello uso el algortimo RSA, que es una especie de montecarlo para generar los discos de manera inicial. Una vez inicializado empieza el MC traslacional. en las simulaciones la distancia que se mueven a cada paso se va reduciendo conforme disminuye el porcentaje de aceptación de los pasos. 

In [ ]:
@njit(fastmath=True)
def init_system(x, y, L, D_eff, head, next_node, ncx, ncy, topology):
    head[:] = -1
    count = 0
    N_total = len(x)
    max_x = L if topology == 0 else 3*L
    max_y = L if topology == 0 else 4*L
    
    while count < N_total:
        px = np.random.uniform(0, max_x)
        py = np.random.uniform(0, max_y)
        
        if not inside(px, py, L, topology): continue
        if not check_overlap(px, py, x, y, D_eff, head, next_node, ncx, ncy, L, topology, -1):
            x[count], y[count] = px, py
            idx = int(px/D_eff) + int(py/D_eff) * ncx
            next_node[count] = head[idx]
            head[idx] = count
            count += 1
    return count

@njit(fastmath=True)
def mc_step(x, y, L, D_eff, max_disp, head, next_node, ncx, ncy, topology):
    N_total = len(x)
    accepts = 0
    for i in range(N_total):
        old_x, old_y = x[i], y[i]
        dx = (np.random.rand() - 0.5) * 2 * max_disp
        dy = (np.random.rand() - 0.5) * 2 * max_disp
        new_x, new_y = PBC(old_x, old_y, old_x + dx, old_y + dy, L, topology)
        
        if not inside(new_x, new_y, L, topology): continue 
            
        if not check_overlap(new_x, new_y, x, y, D_eff, head, next_node, ncx, ncy, L, topology, i):
            x[i], y[i] = new_x, new_y
            accepts += 1
            
            old_idx = int(old_x/D_eff) + int(old_y/D_eff) * ncx
            new_idx = int(new_x/D_eff) + int(new_y/D_eff) * ncx
            if old_idx != new_idx:
                prev, curr = -1, head[old_idx]
                while curr != -1:
                    if curr == i:
                        if prev == -1: head[old_idx] = next_node[curr]
                        else: next_node[prev] = next_node[curr]
                        break
                    prev, curr = curr, next_node[curr]
                next_node[i] = head[new_idx]
                head[new_idx] = i
    return accepts / N_total

# Parte de compresión (MC Volumétrico)

Para llegar al estado de jamming que busco empiezo con una densidad muy baja (ebtorno a 0.1) y a la vez que "agito" las párticulas con el MC de antes, hay que ir comprimiendo la región de simulación y así aumentar la densidad. En los scripts principales 'MC completo (ani)' y 'MC completo (res)' se van alternando ambos MC y comprobando todo el rato si son pasos aceptados o no. Al igual que con el MC traslacional, aquí el parámetro clave es el shrink_factor, que va siendo cada vez más pequeño conforme decae la aceptación para ir aumentando la precisión.

In [ ]:
@njit(fastmath=True)
def compression(x, y, L, D_eff, shrink_factor, head, next_node, ncx, ncy, topology):
    old_x = x.copy()
    old_y = y.copy()
    new_L = L * shrink_factor
    scale_ratio = new_L / L
    
    # contraer coordenadas
    for i in range(len(x)):
        x[i] = x[i] * scale_ratio
        y[i] = y[i] * scale_ratio
        
    # reconstruir Cell Lists con la nueva configuración
    head[:] = -1
    for i in range(len(x)):
        idx = int(x[i]/D_eff) + int(y[i]/D_eff) * ncx
        next_node[i] = head[idx]
        head[idx] = i
        
    # comprobar solapamientos tras la compresión
    overlap = False
    for i in range(len(x)):
        if check_overlap(x[i], y[i], x, y, D_eff, head, next_node, ncx, ncy, new_L, topology, i):
            overlap = True
            break
            
    if overlap:
        # rechazo: restaurar coordenadas si se viola la restricción de solapamiento
        for i in range(len(x)):
            x[i] = old_x[i]
            y[i] = old_y[i]
            
        head[:] = -1
        for i in range(len(x)):
            idx = int(x[i]/D_eff) + int(y[i]/D_eff) * ncx
            next_node[i] = head[idx]
            head[idx] = i
            
        return L, False # retorna el L original y estado rechazado
        
    return new_L, True  # retorna el L densificado y estado aceptado